# NHL Season Analytics — Consistency Rating & Post-Descent Stickiness
### 2010-11 through 2024-25 (13 full 82-game seasons) + 2025-26 in progress

**Core concept:** Every NHL game produces one of three standing point outcomes:
- `2` → Win  |  `1` → OT/SO Loss  |  `0` → Regulation Loss

**Metrics:**
- `SUM(seq)` → Total standings points
- `DESCENTS(seq)` → Count where `seq[i] < seq[i-1]`
- `CONSISTENCY RATING` → z-score vs pooled league baseline, scaled 0-100 within season
- `PDS_0` → Post-Descent Stickiness at 0: after descending to a regulation loss, P(next game is also a regulation loss)
- `PDS_1` → Post-Descent Stickiness at 1: after descending to an OTL, P(next game is also an OTL)

**Conjecture:** High-consistency, high-points teams have higher PDS_0 — after a descent to a loss,
they tend to stay at that level briefly before snapping back into a win streak. This clustering of
losses is precisely what makes them look "consistent" (few descents) in the first place.

**Indicators:** 🏒 `[P]` Playoff team  |  🏆 `[PT]` Presidents Trophy


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':        0.5,
    'font.family':      'DejaVu Sans',
})

BASE_URL    = 'https://api-web.nhle.com/v1'
SEASON_2425 = '20242025'
SEASON_2526 = '20252026'

# 14 full 82-game seasons — skip 2012-13 (48-game lockout)
HISTORICAL_SEASONS = [
    '20102011','20112012','20132014','20142015','20152016',
    '20162017','20172018','20182019','20192020','20202021',
    '20212022','20222023','20232024','20242025'
]
COVID_SEASON = '20192020'   # shortened ~71 games, kept but flagged

PLAYOFFS = {
    '20242025': {'TOR','TBL','FLA','WSH','CAR','NJD','OTT','MTL',
                 'WPG','DAL','COL','VGK','LAK','EDM','MIN','STL'},
    '20252026': {'BUF','TBL','MTL','CAR','PIT','PHI','BOS','OTT',
                 'COL','DAL','MIN','VGK','EDM','ANA','UTA','LAK'},
}
PRESIDENTS = {
    '20242025': {'WPG'},
    '20252026': {'COL'},
}

def get_indicators(abbrev, season):
    badges = []
    if abbrev in PRESIDENTS.get(season, set()):
        badges.append('PT')
    if abbrev in PLAYOFFS.get(season, set()):
        badges.append('P')
    return badges

print("Setup complete.")
print(f"  Historical seasons to fetch: {len(HISTORICAL_SEASONS)}")
print(f"  Expected team-seasons: ~{len(HISTORICAL_SEASONS) * 32}")


In [ ]:
def get_all_teams():
    """
    Fetch all 32 NHL team abbreviations and names from the standings endpoint.
    Returns a list of dicts with keys: abbrev, name, conference, division.
    """
    url = f"{BASE_URL}/standings/now"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    teams = []
    for entry in data['standings']:
        teams.append({
            'abbrev':     entry['teamAbbrev']['default'],
            'name':       entry['teamName']['default'],
            'conference': entry['conferenceName'],
            'division':   entry['divisionName'],
        })

    teams.sort(key=lambda x: x['abbrev'])
    return teams


teams = get_all_teams()
teams_df = pd.DataFrame(teams)

print(f"✅ Found {len(teams)} teams:\n")
print(teams_df.to_string(index=False))


In [ ]:
def get_team_sequence(abbrev, season=SEASON_2425):
    """
    Fetch a team's full season schedule and return a sequence where each
    element is the standings points earned that game:
        2 = Win, 1 = OT/SO Loss, 0 = Regulation Loss
    """
    url  = f"{BASE_URL}/club-schedule-season/{abbrev}/{season}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    sequence = []
    for game in data["games"]:
        if game.get("gameType", 0) != 2:
            continue
        if game.get("gameState", "") not in ("OFF", "FINAL"):
            continue
        home_abbrev = game["homeTeam"]["abbrev"]
        home_score  = game["homeTeam"]["score"]
        away_score  = game["awayTeam"]["score"]
        team_won    = (home_score > away_score) if abbrev == home_abbrev else (away_score > home_score)
        period_type = game.get("periodDescriptor", {}).get("periodType", "REG")
        went_to_ot  = period_type in ("OT", "SO")

        if team_won:
            sequence.append(2)
        elif went_to_ot:
            sequence.append(1)
        else:
            sequence.append(0)

    return sequence


def sum_points(seq):
    return sum(seq)


def count_descents(seq):
    return sum(1 for i in range(1, len(seq)) if seq[i] < seq[i-1])


def post_descent_stickiness(seq, landed_on=0):
    """
    After a descent that lands on `landed_on`, what fraction of the time
    does the NEXT game equal that same value?

    Formally: P(seq[i+1] == landed_on  |  seq[i] == landed_on  AND  seq[i] < seq[i-1])

    This captures whether a team STAYS at the new lower level after a descent
    rather than recovering or declining further — the "clustering" behaviour.

    Returns None if there are no eligible descent positions.
    """
    if len(seq) < 3:
        return None
    eligible = [i for i in range(1, len(seq) - 1)
                if seq[i] < seq[i-1] and seq[i] == landed_on]
    if not eligible:
        return None
    sticky = sum(1 for i in eligible if seq[i+1] == landed_on)
    return sticky / len(eligible)


def descent_recovery_rate(seq):
    """
    After ANY descent at position i, what fraction of the time does
    seq[i+1] > seq[i]? (A genuine improvement — the team bounced back.)
    Complement of stickiness + further decline.
    """
    if len(seq) < 3:
        return None
    eligible = [i for i in range(1, len(seq) - 1) if seq[i] < seq[i-1]]
    if not eligible:
        return None
    recovered = sum(1 for i in eligible if seq[i+1] > seq[i])
    return recovered / len(eligible)


# ── Sanity checks ─────────────────────────────────────────────────────────────
print("Function definitions complete. Sanity checks:")
print()

# Good team: wins streak, then 2 losses, then back to wins
good = [2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 0, 2]
print(f"Good team:  {good}")
print(f"  Descents: {count_descents(good)}")
print(f"  PDS_0:    {post_descent_stickiness(good, 0)}")
print(f"  PDS_1:    {post_descent_stickiness(good, 1)}")
print(f"  Recovery: {descent_recovery_rate(good)}")
print()

# Volatile team: alternating results
vola = [2, 0, 2, 0, 1, 0, 2, 0, 2, 0, 1, 0]
print(f"Volatile:   {vola}")
print(f"  Descents: {count_descents(vola)}")
print(f"  PDS_0:    {post_descent_stickiness(vola, 0)}")
print(f"  PDS_1:    {post_descent_stickiness(vola, 1)}")
print(f"  Recovery: {descent_recovery_rate(vola)}")
print()

# Bad consistent team: long losing streaks
bad  = [0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0]
print(f"Bad consis: {bad}")
print(f"  Descents: {count_descents(bad)}")
print(f"  PDS_0:    {post_descent_stickiness(bad, 0)}")
print(f"  PDS_1:    {post_descent_stickiness(bad, 1)}")
print(f"  Recovery: {descent_recovery_rate(bad)}")


In [ ]:
# Fetches all historical seasons + 2025-26
# ~8-12 minutes with rate limiting

all_sequences = {}
fetch_errors  = []

for season in HISTORICAL_SEASONS + [SEASON_2526]:
    all_sequences[season] = {}
    label = f"{season[:4]}-{season[4:6]}"
    print(f"Fetching {label}...", end=" ", flush=True)

    for team in teams:
        abbrev = team["abbrev"]
        if abbrev == "UTA" and season < "20242025":
            continue   # Utah Mammoth did not exist before 2024-25
        try:
            seq = get_team_sequence(abbrev, season)
            if len(seq) > 0:
                all_sequences[season][abbrev] = seq
        except Exception as e:
            fetch_errors.append((season, abbrev, str(e)))
        time.sleep(0.12)

    n_teams = len(all_sequences[season])
    n_games = sum(len(s) for s in all_sequences[season].values())
    print(f"{n_teams} teams, {n_games} games")

total_ts = sum(len(v) for v in all_sequences.values())
total_g  = sum(len(s) for v in all_sequences.values() for s in v.values())
print(f"\nFetch complete! {total_ts} team-seasons, {total_g} total games.")
if fetch_errors:
    print(f"Errors: {fetch_errors[:5]}")

# Alias so later cells still work
sequences = all_sequences[SEASON_2425]


In [ ]:
# ── Pooled baseline from all full 82-game historical seasons ─────────────────
baseline_results = []
for season in HISTORICAL_SEASONS:
    if season == COVID_SEASON:
        continue
    for seq in all_sequences.get(season, {}).values():
        if len(seq) >= 75:
            baseline_results.extend(seq)

total_bl         = len(baseline_results)
bp0              = baseline_results.count(0) / total_bl
bp1              = baseline_results.count(1) / total_bl
bp2              = baseline_results.count(2) / total_bl
p_descent_pooled = bp0*bp1 + bp0*bp2 + bp1*bp2
e_desc_pooled    = 81 * p_descent_pooled
std_desc_pooled  = np.sqrt(81 * p_descent_pooled * (1 - p_descent_pooled))

print("Pooled Baseline")
print(f"  Games: {total_bl}")
print(f"  P(0)={bp0:.4f}  P(1)={bp1:.4f}  P(2)={bp2:.4f}")
print(f"  P(descent)={p_descent_pooled:.4f}  E[descents]={e_desc_pooled:.2f}  Std={std_desc_pooled:.2f}")


# ── Build master DataFrame ────────────────────────────────────────────────────
def build_master():
    rows      = []
    team_meta = {t["abbrev"]: t for t in teams}

    for season in HISTORICAL_SEASONS + [SEASON_2526]:
        label    = f"{season[:4]}-{season[4:6]}"
        is_covid = (season == COVID_SEASON)

        for abbrev, seq in all_sequences.get(season, {}).items():
            if len(seq) < 30:
                continue
            meta   = team_meta.get(abbrev, {})
            desc   = count_descents(seq)
            pds0   = post_descent_stickiness(seq, landed_on=0)
            pds1   = post_descent_stickiness(seq, landed_on=1)
            drr    = descent_recovery_rate(seq)
            z      = (desc - e_desc_pooled) / std_desc_pooled
            badges = get_indicators(abbrev, season)

            rows.append({
                "season":       season,
                "season_label": label,
                "abbrev":       abbrev,
                "name":         meta.get("name", abbrev),
                "conference":   meta.get("conference", ""),
                "games":        len(seq),
                "points":       sum_points(seq),
                "descents":     desc,
                "pds0":         pds0,
                "pds1":         pds1,
                "drr":          drr,
                "z_score":      z,
                "covid":        is_covid,
                "playoff":      "P" in badges,
                "presidents":   "PT" in badges,
                "in_progress":  (season == SEASON_2526),
            })

    df = pd.DataFrame(rows)

    # Scale CR 0-100 within each season
    for season in df["season"].unique():
        mask  = df["season"] == season
        z_min = df.loc[mask, "z_score"].min()
        z_max = df.loc[mask, "z_score"].max()
        if z_max > z_min:
            df.loc[mask, "consistency_rating"] = 100 * (
                1 - (df.loc[mask, "z_score"] - z_min) / (z_max - z_min)
            )
        else:
            df.loc[mask, "consistency_rating"] = 50.0

    return df


master = build_master()

# Backward-compatible aliases
summary_2425  = master[master["season"] == SEASON_2425].copy().reset_index(drop=True)
summary_2526  = master[master["season"] == SEASON_2526].copy().reset_index(drop=True)
summary       = summary_2425
expected_desc = e_desc_pooled
std_desc      = std_desc_pooled
exp_2425 = exp_2526 = e_desc_pooled
std_2425 = std_2526 = std_desc_pooled

print(f"\nMaster dataset: {len(master)} team-seasons, {master['season'].nunique()} seasons")
print(f"PDS_0 available: {master['pds0'].notna().sum()} team-seasons")
print(f"PDS_1 available: {master['pds1'].notna().sum()} team-seasons")
print(f"DRR available:   {master['drr'].notna().sum()} team-seasons")


In [ ]:
def team_color(conference):
    """Return a fill color by conference."""
    return '#1f6feb' if conference == 'Eastern' else '#f78166'


def badge_label(abbrev, season):
    """Build compact badge string for chart title."""
    badges = get_indicators(abbrev, season)
    parts = []
    if 'PT' in badges:
        parts.append('🏆PT')
    if 'P' in badges:
        parts.append('🏒P')
    return '  ' + '  '.join(parts) if parts else ''


def plot_team_sequence(ax, abbrev, seq, meta, season, summary_df):
    """
    Bar chart for one team's season sequence with indicator markers.
    """
    games  = list(range(1, len(seq) + 1))
    color  = team_color(meta['conference'])
    pts    = sum_points(seq)
    descs  = count_descents(seq)
    cr_row = summary_df.loc[summary_df['abbrev'] == abbrev, 'consistency_rating']
    cr     = cr_row.values[0] if len(cr_row) else 0
    badges = get_indicators(abbrev, season)

    bar_colors = []
    for v in seq:
        if v == 2:
            bar_colors.append(color)
        elif v == 1:
            bar_colors.append('#8b949e')
        else:
            bar_colors.append('#21262d')

    ax.bar(games, seq, color=bar_colors, width=1.0, align='center',
           edgecolor='none', zorder=3)

    # ── Indicator markers ─────────────────────────────────────────────────────
    last_game = len(seq)

    # President's Trophy: gold star above last bar
    if 'PT' in badges and seq:
        ax.annotate('★', xy=(last_game, seq[-1]),
                    xytext=(last_game, 2.05),
                    fontsize=7, color='#FFD700', ha='center', va='bottom',
                    annotation_clip=False)

    # Playoff: draw a subtle green top border on the axes
    if 'P' in badges:
        ax.spines['top'].set_color('#3fb950')
        ax.spines['top'].set_linewidth(2.0)
    else:
        ax.spines['top'].set_color('#30363d')
        ax.spines['top'].set_linewidth(0.5)

    ax.set_xlim(0.5, 82.5)
    ax.set_ylim(0, 2)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['0', '1', '2'], fontsize=6)
    ax.tick_params(axis='x', labelsize=6)
    ax.grid(True, axis='y', alpha=0.3, zorder=0)

    badge_str = badge_label(abbrev, season)
    ax.set_title(
        f"{abbrev}  |  {pts}pts  |  {descs}d  |  CR{cr:.0f}{badge_str}",
        fontsize=6.5, pad=3, color='#e6edf3'
    )


def plot_season_grid(season_key, seq_dict, summary_df, title):
    """Render the full 32-team grid for a given season."""
    sorted_abbrevs = sorted(seq_dict.keys())
    n     = len(sorted_abbrevs)
    ncols = 4
    nrows = -(-n // ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 2.4))
    fig.suptitle(title, fontsize=15, y=1.01, color='#e6edf3', fontweight='bold')
    fig.patch.set_facecolor('#0d1117')

    team_meta = {t['abbrev']: t for t in teams}

    for idx, abbrev in enumerate(sorted_abbrevs):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]
        ax.set_facecolor('#0d1117')
        plot_team_sequence(ax, abbrev, seq_dict[abbrev], team_meta[abbrev], season_key, summary_df)

    for idx in range(n, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    # Legend
    east_patch  = mpatches.Patch(color='#1f6feb', alpha=0.9, label='Eastern — Win')
    west_patch  = mpatches.Patch(color='#f78166', alpha=0.9, label='Western — Win')
    otl_patch   = mpatches.Patch(color='#8b949e', alpha=0.9, label='OT/SO Loss')
    loss_patch  = mpatches.Patch(color='#21262d', alpha=0.9, label='Reg Loss')
    play_patch  = mpatches.Patch(color='#3fb950', alpha=0.9, label='🏒 Playoff team (green border)')
    pt_text     = mpatches.Patch(color='#FFD700', alpha=0.9, label='🏆 Presidents Trophy (★ last bar)')
    fig.legend(handles=[east_patch, west_patch, otl_patch, loss_patch, play_patch, pt_text],
               loc='lower center', ncol=6, fontsize=8,
               framealpha=0.3, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    return fig


# ── Render 2024-25 ────────────────────────────────────────────────────────────
fig_2425 = plot_season_grid(
    SEASON_2425, all_sequences[SEASON_2425], summary_2425,
    "NHL 2024-25 Season — Per-Game Standing Points"
)
fig_2425.savefig('nhl_sequences_2425.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_sequences_2425.png")

# ── Render 2025-26 ────────────────────────────────────────────────────────────
fig_2526 = plot_season_grid(
    SEASON_2526, all_sequences[SEASON_2526], summary_2526,
    "NHL 2025-26 Season — Per-Game Standing Points (In Progress)"
)
fig_2526.savefig('nhl_sequences_2526.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_sequences_2526.png")


In [ ]:
def plot_consistency_scatter(summary_df, season_label, expected, std):
    """Points vs Consistency Rating scatter with indicator callouts."""
    fig, ax = plt.subplots(figsize=(13, 7))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    for _, row in summary_df.iterrows():
        color = '#1f6feb' if row['conference'] == 'Eastern' else '#f78166'
        # Playoff teams: brighter, larger marker
        size    = 140 if row['playoff'] else 70
        alpha   = 1.0 if row['playoff'] else 0.5
        marker  = '*' if row['presidents'] else ('o' if row['playoff'] else 's')
        ax.scatter(row['points'], row['consistency_rating'],
                   c=color, s=size, alpha=alpha, marker=marker, zorder=3,
                   edgecolors='#FFD700' if row['presidents'] else 'none',
                   linewidths=1.5)
        ax.annotate(row['abbrev'],
                    (row['points'], row['consistency_rating']),
                    textcoords='offset points', xytext=(6, 3),
                    fontsize=7.5, color='#e6edf3', alpha=0.9)

    med_pts = summary_df['points'].median()
    med_cr  = summary_df['consistency_rating'].median()
    ax.axvline(med_pts, color='#8b949e', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(med_cr,  color='#8b949e', linestyle='--', alpha=0.5, linewidth=1)

    pts_min = summary_df['points'].min()
    pts_max = summary_df['points'].max()
    ax.text(pts_min + 1, 96, 'Low pts / Consistent',  fontsize=8, color='#8b949e', alpha=0.7)
    ax.text(pts_max - 16, 96, 'High pts / Consistent', fontsize=8, color='#8b949e', alpha=0.7)
    ax.text(pts_min + 1, 4,  'Low pts / Volatile',    fontsize=8, color='#8b949e', alpha=0.7)
    ax.text(pts_max - 16, 4,  'High pts / Volatile',   fontsize=8, color='#8b949e', alpha=0.7)

    ax.set_xlabel('Season Standings Points', fontsize=12)
    ax.set_ylabel('Consistency Rating (0=volatile, 100=consistent)', fontsize=12)
    ax.set_title(f'NHL {season_label}: Points vs Consistency Rating', fontsize=14,
                 fontweight='bold', pad=12)

    east_patch  = mpatches.Patch(color='#1f6feb', alpha=0.8, label='Eastern Conf')
    west_patch  = mpatches.Patch(color='#f78166', alpha=0.8, label='Western Conf')
    play_patch  = plt.scatter([], [], c='white', s=130, alpha=1.0, marker='o',
                               edgecolors='none', label='🏒 Playoff team (large)')
    nonplay     = plt.scatter([], [], c='white', s=60, alpha=0.4, marker='s',
                               edgecolors='none', label='Non-playoff (small/dim)')
    pt_patch    = plt.scatter([], [], c='white', s=200, alpha=1.0, marker='*',
                               edgecolors='#FFD700', linewidths=1.5,
                               label='🏆 Presidents Trophy (gold border ★)')
    ax.legend(handles=[east_patch, west_patch, play_patch, nonplay, pt_patch],
              fontsize=9, framealpha=0.3)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig


fig_s1 = plot_consistency_scatter(summary_2425, '2024-25', exp_2425, std_2425)
fig_s1.savefig('nhl_scatter_2425.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_scatter_2425.png")

fig_s2 = plot_consistency_scatter(summary_2526, '2025-26', exp_2526, std_2526)
fig_s2.savefig('nhl_scatter_2526.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_scatter_2526.png")


In [ ]:
def plot_descent_bars(summary_df, season_label, expected, std):
    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    sorted_df  = summary_df.sort_values('descents')
    bar_colors = []
    edge_colors = []
    edge_widths = []
    for _, row in sorted_df.iterrows():
        base = '#1f6feb' if row['conference'] == 'Eastern' else '#f78166'
        bar_colors.append(base)
        if row['presidents']:
            edge_colors.append('#FFD700')
            edge_widths.append(2.0)
        elif row['playoff']:
            edge_colors.append('#3fb950')
            edge_widths.append(1.5)
        else:
            edge_colors.append('none')
            edge_widths.append(0)

    bars = ax.bar(sorted_df['abbrev'], sorted_df['descents'],
                  color=bar_colors, alpha=0.8,
                  edgecolor=edge_colors, linewidth=edge_widths, zorder=3)

    ax.axhline(expected, color='#f0e68c', linestyle='-', linewidth=1.8,
               label=f'E[descents] = {expected:.1f}', zorder=4)
    ax.axhspan(expected - std, expected + std,
               alpha=0.12, color='#f0e68c', label=f'±1 std ({std:.1f})')
    ax.axhspan(expected - 2*std, expected + 2*std,
               alpha=0.06, color='#f0e68c', label=f'±2 std')

    ax.set_xlabel('Team', fontsize=11)
    ax.set_ylabel('Descent Count', fontsize=11)
    ax.set_title(f'NHL {season_label}: Descent Count vs League-Expected',
                 fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(True, axis='y', alpha=0.3)

    east_p = mpatches.Patch(color='#1f6feb', alpha=0.8, label='Eastern Conf')
    west_p = mpatches.Patch(color='#f78166', alpha=0.8, label='Western Conf')
    play_p = mpatches.Patch(edgecolor='#3fb950', facecolor='none',
                             linewidth=1.5, label='🏒 Playoff (green border)')
    pt_p   = mpatches.Patch(edgecolor='#FFD700', facecolor='none',
                             linewidth=2.0, label='🏆 Presidents Trophy (gold border)')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [east_p, west_p, play_p, pt_p],
              fontsize=8, framealpha=0.3)

    plt.tight_layout()
    return fig


fig_d1 = plot_descent_bars(summary_2425, '2024-25', exp_2425, std_2425)
fig_d1.savefig('nhl_descents_2425.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_descents_2425.png")

fig_d2 = plot_descent_bars(summary_2526, '2025-26', exp_2526, std_2526)
fig_d2.savefig('nhl_descents_2526.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: nhl_descents_2526.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# POST-DESCENT STICKINESS ANALYSIS
#
# Conjecture: High-consistency, high-points teams have higher PDS_0
# meaning after a descent to a loss they tend to stay there briefly
# before snapping back — this clustering IS their consistency.
# ═══════════════════════════════════════════════════════════════════════════

# Full 82-game seasons only (no covid, no in-progress)
pds_df = master[
    (~master["covid"]) &
    (~master["in_progress"]) &
    (master["games"] >= 75) &
    (master["pds0"].notna())
].copy().reset_index(drop=True)

print(f"Team-seasons in PDS analysis: {len(pds_df)}")
print(f"PDS_0 — min:{pds_df['pds0'].min():.3f}  max:{pds_df['pds0'].max():.3f}  mean:{pds_df['pds0'].mean():.3f}")
print(f"PDS_1 — available for {master[(~master['covid']) & (~master['in_progress']) & master['pds1'].notna()].shape[0]} team-seasons")

# ── Correlations ──────────────────────────────────────────────────────────────
r_cr,  p_cr  = stats.pearsonr(pds_df["consistency_rating"], pds_df["pds0"])
r_pts, p_pts = stats.pearsonr(pds_df["points"],             pds_df["pds0"])

print(f"\nCorrelations with PDS_0:")
print(f"  CR  vs PDS_0:  r={r_cr:.4f}   p={p_cr:.4f}   {'Significant' if p_cr  < 0.05 else 'Not significant'}")
print(f"  Pts vs PDS_0:  r={r_pts:.4f}  p={p_pts:.4f}  {'Significant' if p_pts < 0.05 else 'Not significant'}")

# ── Top 16 by consistency rating ──────────────────────────────────────────────
top16 = pds_df.nlargest(16, "consistency_rating")[[
    "season_label","abbrev","points","descents",
    "consistency_rating","pds0","pds1","drr","playoff","presidents"
]].reset_index(drop=True)
top16.index += 1

print(f"\nTop 16 Most Consistent Team-Seasons:")
print(top16.to_string())

# ── T-test: top 16 PDS_0 vs rest ──────────────────────────────────────────────
top16_idx = pds_df.nlargest(16, "consistency_rating").index
rest_pds0 = pds_df.loc[~pds_df.index.isin(top16_idx), "pds0"].dropna()
top16_pds0 = pds_df.loc[pds_df.index.isin(top16_idx), "pds0"].dropna()
t_stat, t_pval = stats.ttest_ind(top16_pds0, rest_pds0)

print(f"\nTop 16 avg PDS_0:       {top16_pds0.mean():.4f}")
print(f"Rest of dataset PDS_0:  {rest_pds0.mean():.4f}")
print(f"T-test:  t={t_stat:.3f}  p={t_pval:.4f}  {'Significant' if t_pval < 0.05 else 'Not significant'}")

# ── Quadrant analysis: high CR + high points ──────────────────────────────────
med_cr  = pds_df["consistency_rating"].median()
med_pts = pds_df["points"].median()

q_high_both = pds_df[(pds_df["consistency_rating"] > med_cr) & (pds_df["points"] > med_pts)]
q_high_cr   = pds_df[(pds_df["consistency_rating"] > med_cr) & (pds_df["points"] <= med_pts)]
q_high_pts  = pds_df[(pds_df["consistency_rating"] <= med_cr) & (pds_df["points"] > med_pts)]
q_low_both  = pds_df[(pds_df["consistency_rating"] <= med_cr) & (pds_df["points"] <= med_pts)]

print(f"\nPDS_0 by quadrant (median CR={med_cr:.0f}, median pts={med_pts:.0f}):")
print(f"  High CR + High Pts (n={len(q_high_both):>3}):  PDS_0 = {q_high_both['pds0'].mean():.4f}  ← The conjecture targets this group")
print(f"  High CR + Low Pts  (n={len(q_high_cr):>3}):  PDS_0 = {q_high_cr['pds0'].mean():.4f}")
print(f"  Low  CR + High Pts (n={len(q_high_pts):>3}):  PDS_0 = {q_high_pts['pds0'].mean():.4f}")
print(f"  Low  CR + Low Pts  (n={len(q_low_both):>3}):  PDS_0 = {q_low_both['pds0'].mean():.4f}")


In [ ]:
# ── 2x2 scatter grid: CR vs PDS_0, Pts vs PDS_0, and quadrant map ────────────
top16_idx = pds_df.nlargest(16, "consistency_rating").index

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor("#0d1117")
fig.suptitle(
    "Post-Descent Stickiness at 0 (PDS_0) Analysis\n"
    "After descending to a regulation loss, P(next game is also a regulation loss)",
    fontsize=13, fontweight="bold", color="#e6edf3", y=1.01
)

def scatter_with_regression(ax, x_col, y_col, xlabel, title):
    """Helper: scatter + regression + top16 labels."""
    slope, intercept, r_val, p_val, se = stats.linregress(
        pds_df[x_col], pds_df[y_col]
    )
    for idx, row in pds_df.iterrows():
        color  = "#1f6feb" if row["conference"] == "Eastern" else "#f78166"
        is_top = idx in top16_idx
        ax.scatter(row[x_col], row[y_col],
                   c=color, s=120 if is_top else 28,
                   alpha=0.95 if is_top else 0.22,
                   edgecolors="#FFD700" if row["presidents"] else "none",
                   linewidths=1.2, zorder=4 if is_top else 2)
        if is_top:
            ax.annotate(f"{row['abbrev']} {row['season_label']}",
                        (row[x_col], row[y_col]),
                        textcoords="offset points", xytext=(5, 3),
                        fontsize=6.5, color="#e6edf3", zorder=5)

    x_line = np.linspace(pds_df[x_col].min(), pds_df[x_col].max(), 200)
    y_line = slope * x_line + intercept
    n      = len(pds_df)
    x_mean = pds_df[x_col].mean()
    ss_x   = ((pds_df[x_col] - x_mean)**2).sum()
    se_b   = se * np.sqrt(1/n + (x_line - x_mean)**2 / ss_x)
    ax.fill_between(x_line, y_line - 1.96*se_b, y_line + 1.96*se_b,
                    alpha=0.15, color="#f0e68c")
    ax.plot(x_line, y_line, color="#f0e68c", linewidth=2,
            label=f"r={r_val:.3f}  p={p_val:.4f}")
    ax.axvline(pds_df[x_col].median(), color="#8b949e",
               linestyle="--", alpha=0.4, linewidth=1)
    ax.axhline(pds_df[y_col].median(), color="#8b949e",
               linestyle="--", alpha=0.4, linewidth=1)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("PDS_0 (Post-Descent Stickiness at 0)", fontsize=10)
    ax.set_title(title, fontsize=10, color="#e6edf3")
    ax.legend(fontsize=9, framealpha=0.3)
    ax.grid(True, alpha=0.25)
    ax.set_facecolor("#0d1117")
    return r_val, p_val

for ax in axes.flat:
    ax.set_facecolor("#0d1117")

# Top-left: CR vs PDS_0
r1, p1 = scatter_with_regression(
    axes[0][0], "consistency_rating", "pds0",
    "Consistency Rating (0=volatile, 100=consistent)",
    "Consistency Rating vs PDS_0"
)

# Top-right: Points vs PDS_0
r2, p2 = scatter_with_regression(
    axes[0][1], "points", "pds0",
    "Season Standings Points",
    "Season Points vs PDS_0"
)

# Bottom-left: PDS_1 vs CR (secondary metric)
pds1_df = master[
    (~master["covid"]) & (~master["in_progress"]) &
    (master["games"] >= 75) & (master["pds1"].notna())
].copy().reset_index(drop=True)

ax3 = axes[1][0]
ax3.set_facecolor("#0d1117")
if len(pds1_df) > 10:
    slope3, intercept3, r3, p3, se3 = stats.linregress(
        pds1_df["consistency_rating"], pds1_df["pds1"]
    )
    colors3 = ["#1f6feb" if c == "Eastern" else "#f78166"
               for c in pds1_df["conference"]]
    ax3.scatter(pds1_df["consistency_rating"], pds1_df["pds1"],
                c=colors3, s=28, alpha=0.35, zorder=2)
    x3 = np.linspace(pds1_df["consistency_rating"].min(),
                      pds1_df["consistency_rating"].max(), 200)
    ax3.plot(x3, slope3 * x3 + intercept3, color="#f0e68c", linewidth=2,
             label=f"r={r3:.3f}  p={p3:.4f}")
    ax3.set_xlabel("Consistency Rating", fontsize=10)
    ax3.set_ylabel("PDS_1 (Stickiness at OTL)", fontsize=10)
    ax3.set_title("Consistency Rating vs PDS_1 (OTL stickiness)", fontsize=10, color="#e6edf3")
    ax3.legend(fontsize=9, framealpha=0.3)
    ax3.grid(True, alpha=0.25)
else:
    ax3.text(0.5, 0.5, "Insufficient PDS_1 data", ha="center", va="center",
             transform=ax3.transAxes, color="#8b949e")

# Bottom-right: Quadrant heatmap — avg PDS_0 by CR quartile x Points quartile
ax4 = axes[1][1]
ax4.set_facecolor("#0d1117")
pds_df2 = pds_df.copy()
pds_df2["cr_q"]  = pd.qcut(pds_df2["consistency_rating"], 4,
                             labels=["Q1
(volatile)","Q2","Q3","Q4
(consistent)"])
pds_df2["pts_q"] = pd.qcut(pds_df2["points"], 4,
                             labels=["Q1
(low pts)","Q2","Q3","Q4
(high pts)"])
pivot = pds_df2.pivot_table(values="pds0", index="pts_q", columns="cr_q", aggfunc="mean")
im = ax4.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto",
                vmin=0, vmax=pds_df["pds0"].max())
ax4.set_xticks(range(len(pivot.columns)))
ax4.set_xticklabels(pivot.columns, fontsize=8)
ax4.set_yticks(range(len(pivot.index)))
ax4.set_yticklabels(pivot.index, fontsize=8)
ax4.set_xlabel("Consistency Rating Quartile", fontsize=10)
ax4.set_ylabel("Points Quartile", fontsize=10)
ax4.set_title("Avg PDS_0 by Points × Consistency Quartile
(red=high stickiness, green=low)", fontsize=9, color="#e6edf3")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax4.text(j, i, f"{val:.3f}", ha="center", va="center",
                     fontsize=9, color="white", fontweight="bold")
plt.colorbar(im, ax=ax4, label="Avg PDS_0", shrink=0.8)

plt.tight_layout()
plt.savefig("nhl_pds_analysis.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: nhl_pds_analysis.png")


In [ ]:
# ── Top 16 bar chart + PDS_0 distribution comparison ────────────────────────
top16_idx  = pds_df.nlargest(16, "consistency_rating").index
top16      = pds_df.loc[top16_idx].reset_index(drop=True)
top16.index += 1
rest_pds0  = pds_df.loc[~pds_df.index.isin(top16_idx), "pds0"].dropna()
t_stat, t_pval = stats.ttest_ind(top16["pds0"].dropna(), rest_pds0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("Top 16 Most Consistent NHL Team-Seasons — PDS_0 Deep Dive",
             fontsize=14, fontweight="bold", color="#e6edf3")

# Left: horizontal bars ranked by CR, coloured by PDS_0
ax1.set_facecolor("#0d1117")
labels   = [f"{r['abbrev']}  {r['season_label']}" for _, r in top16.iterrows()]
cr_vals  = top16["consistency_rating"].values
pds_vals = top16["pds0"].fillna(0).values
norm     = plt.Normalize(0, pds_df["pds0"].max())
cmap     = plt.cm.RdYlGn_r
colors   = [cmap(norm(v)) for v in pds_vals]

ax1.barh(range(len(labels)), cr_vals, color=colors, alpha=0.85, zorder=3)
for i, (cr, pds, pts, playoff) in enumerate(
        zip(cr_vals, pds_vals, top16["points"].values, top16["playoff"].values)):
    marker = "🏒" if playoff else "  "
    ax1.text(cr + 0.3, i,
             f"PDS_0={pds:.3f}  {pts}pts {marker}",
             va="center", fontsize=8, color="#e6edf3")

ax1.set_yticks(range(len(labels)))
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel("Consistency Rating", fontsize=10)
ax1.set_title("Ranked by CR  (bar color = PDS_0: red=sticky, green=bounces back)",
              fontsize=9, color="#e6edf3")
ax1.grid(True, axis="x", alpha=0.3)
ax1.invert_yaxis()
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax1, label="PDS_0", shrink=0.7)

# Right: PDS_0 density — top 16 vs full dataset vs high-pts-high-CR quadrant
ax2.set_facecolor("#0d1117")
med_cr  = pds_df["consistency_rating"].median()
med_pts = pds_df["points"].median()
elite   = pds_df[(pds_df["consistency_rating"] > med_cr) & (pds_df["points"] > med_pts)]["pds0"].dropna()

ax2.hist(pds_df["pds0"].dropna(), bins=20, color="#1f6feb", alpha=0.35,
         label=f"All ({len(pds_df)})", density=True)
ax2.hist(elite, bins=15, color="#3fb950", alpha=0.5,
         label=f"High CR + High Pts (n={len(elite)})", density=True)
ax2.hist(top16["pds0"].dropna(), bins=10, color="#f78166", alpha=0.7,
         label=f"Top 16 CR", density=True)

ax2.axvline(pds_df["pds0"].mean(),  color="#1f6feb", linestyle="--", linewidth=1.5,
            label=f"All mean={pds_df['pds0'].mean():.3f}")
ax2.axvline(elite.mean(),            color="#3fb950", linestyle="--", linewidth=1.5,
            label=f"Elite mean={elite.mean():.3f}")
ax2.axvline(top16["pds0"].mean(),   color="#f78166", linestyle="--", linewidth=1.5,
            label=f"Top16 mean={top16['pds0'].mean():.3f}")

ax2.set_xlabel("PDS_0", fontsize=10)
ax2.set_ylabel("Density", fontsize=10)
ax2.set_title(
    f"PDS_0 Distribution\n"
    f"Top16 vs Elite vs All  |  t={t_stat:.2f}  p={t_pval:.4f}  "
    f"({'Significant' if t_pval < 0.05 else 'Not significant'})",
    fontsize=9, color="#e6edf3"
)
ax2.legend(fontsize=8.5, framealpha=0.3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("nhl_top16_pds.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

# Final conjecture verdict
print("\n Conjecture Assessment")
print(f"  CR  vs PDS_0:  r={r_cr:.4f}  p={p_cr:.4f}")
print(f"  Pts vs PDS_0:  r={r_pts:.4f}  p={p_pts:.4f}")
print(f"  Top 16 PDS_0 mean:     {top16['pds0'].mean():.4f}")
print(f"  Elite quadrant mean:   {elite.mean():.4f}")
print(f"  Full dataset mean:     {pds_df['pds0'].mean():.4f}")
print(f"  T-test p (top16 vs rest): {t_pval:.4f}")
print()
if p_cr < 0.05 and r_cr > 0:
    print("  CONJECTURE SUPPORTED: Consistent teams show higher PDS_0 —")
    print("  after descending to a loss they tend to cluster losses before")
    print("  snapping back, which is the mechanism behind their consistency.")
elif p_cr < 0.05 and r_cr < 0:
    print("  INVERSE FINDING: Consistent teams actually recover FASTER after")
    print("  a descent — their losses do not cluster. Volatility drives loss clustering.")
else:
    print("  INCONCLUSIVE: No significant relationship found.")
print("\nSaved: nhl_top16_pds.png")


In [ ]:
# ── Theoretical expected descents over all possible length-82 strings ─────────
#
# For any two adjacent positions i-1, i drawn i.i.d. from {0,1,2}
# P(descent) = P(seq[i] < seq[i-1])
#
# Using empirical probabilities p0, p1, p2 derived from 2024-25 data:
#   P(descent) = P(curr=0, prev=1) + P(curr=0, prev=2) + P(curr=1, prev=2)
#              = p0*p1 + p0*p2 + p1*p2
#
# For uniform {0,1,2}: p0=p1=p2=1/3 → P(descent) = 1/3 → E = 27.0
#
# We already computed the empirical version above.
# Here we also compute the uniform baseline for comparison.

p_desc_uniform = 1/3
e_desc_uniform = 81 * p_desc_uniform
std_desc_uniform = np.sqrt(81 * p_desc_uniform * (1 - p_desc_uniform))

print("══ Expected Descents — Theoretical Analysis ════════════════════════════")
print()
print("  Uniform distribution (p0=p1=p2=1/3):")
print(f"    P(descent)   = {p_desc_uniform:.4f}")
print(f"    E[descents]  = {e_desc_uniform:.2f}")
print(f"    Std          = {std_desc_uniform:.2f}")
print()
print("  Empirical 2024-25 distribution:")
print(f"    P(0)={p0:.3f}, P(1)={p1:.3f}, P(2)={p2:.3f}")
print(f"    P(descent)   = {p_descent:.4f}")
print(f"    E[descents]  = {expected_desc:.2f}")
print(f"    Std          = {std_desc:.2f}")
print()
print("  Note: The empirical baseline is preferred for z-score comparisons")
print("  as it reflects the true distribution of NHL game outcomes.")
print()
print("── Z-scores (negative = fewer descents than expected = MORE consistent) ─")
print()
print(f"  {'Team':<6} {'Descents':>8} {'Z-Score':>9} {'Interpretation'}")
print(f"  {'────':<6} {'────────':>8} {'───────':>9} {'───────────────────────────'}")
for _, row in summary.sort_values('z_score').iterrows():
    if row['z_score'] <= -1.5:
        interp = "⬛ Very consistent (streaky)"
    elif row['z_score'] <= -0.5:
        interp = "🔵 Somewhat consistent"
    elif row['z_score'] <= 0.5:
        interp = "⚪ Near average"
    elif row['z_score'] <= 1.5:
        interp = "🟠 Somewhat volatile"
    else:
        interp = "🔴 Very volatile"
    print(f"  {row['abbrev']:<6} {row['descents']:>8}   {row['z_score']:>+7.2f}   {interp}")


In [ ]:
def deep_dive(abbrev, season=SEASON_2425):
    """
    Full breakdown for a single team and season:
    - Bar chart with descent highlights + indicator markers
    - Cumulative points vs pace lines
    """
    seq_dict   = all_sequences[season]
    summary_df = summary_2425 if season == SEASON_2425 else summary_2526
    label      = '2024-25' if season == SEASON_2425 else '2025-26'

    if abbrev not in seq_dict:
        print(f"{abbrev} not found for {label}")
        return

    seq    = seq_dict[abbrev]
    meta   = {t['abbrev']: t for t in teams}[abbrev]
    pts    = sum_points(seq)
    desc   = count_descents(seq)
    cr_row = summary_df.loc[summary_df['abbrev'] == abbrev, 'consistency_rating']
    cr     = cr_row.values[0] if len(cr_row) else 0
    z_row  = summary_df.loc[summary_df['abbrev'] == abbrev, 'z_score']
    z      = z_row.values[0] if len(z_row) else 0
    color  = team_color(meta['conference'])
    games  = list(range(1, len(seq) + 1))
    cumulative = np.cumsum(seq)
    badges = get_indicators(abbrev, season)
    badge_str = badge_label(abbrev, season)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle(
        f"{meta['name']} ({abbrev})  —  {label} Season{badge_str}\n"
        f"{pts} pts  |  {desc} descents  |  CR {cr:.0f}  |  z = {z:+.2f}",
        fontsize=13, fontweight='bold', color='#e6edf3'
    )

    # ── Top: bar chart ────────────────────────────────────────────────────────
    ax1.set_facecolor('#0d1117')
    descent_indices = set(i for i in range(1, len(seq)) if seq[i] < seq[i-1])

    bar_colors = []
    for i, v in enumerate(seq):
        if i in descent_indices:
            bar_colors.append('#ff7b72')
        elif v == 2:
            bar_colors.append(color)
        elif v == 1:
            bar_colors.append('#8b949e')
        else:
            bar_colors.append('#21262d')

    ax1.bar(games, seq, color=bar_colors, width=1.0, align='center',
            edgecolor='none', zorder=3)

    # President's Trophy star marker on last bar
    if 'PT' in badges and seq:
        last = len(seq)
        ax1.annotate('★ Presidents Trophy', xy=(last, seq[-1]),
                     xytext=(last - 10, 1.85), fontsize=9,
                     color='#FFD700', fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='#FFD700', lw=1.2))

    # Playoff border
    if 'P' in badges:
        ax1.spines['top'].set_color('#3fb950')
        ax1.spines['top'].set_linewidth(3.0)
        ax1.spines['right'].set_color('#3fb950')
        ax1.spines['right'].set_linewidth(3.0)

    ax1.set_ylim(0, 2)
    ax1.set_yticks([0, 1, 2])
    ax1.set_ylabel('Points Earned', fontsize=10)
    ax1.grid(True, axis='y', alpha=0.3, zorder=0)

    win_p  = mpatches.Patch(color=color,     label='Win')
    otl_p  = mpatches.Patch(color='#8b949e', label='OT/SO Loss')
    loss_p = mpatches.Patch(color='#21262d', label='Reg Loss')
    desc_p = mpatches.Patch(color='#ff7b72', label=f'Descent ({desc})')
    ax1.legend(handles=[win_p, otl_p, loss_p, desc_p],
               fontsize=9, framealpha=0.3, loc='upper right')

    # ── Bottom: cumulative points ─────────────────────────────────────────────
    ax2.set_facecolor('#0d1117')
    pace_line   = [2 * g for g in games]
    ax2.plot(games, pace_line, color='#8b949e', linestyle=':', linewidth=1,
             alpha=0.5, label='2pt pace (max)')

    league_ppg  = sum(sum(s) for s in seq_dict.values()) / sum(len(s) for s in seq_dict.values())
    league_pace = [league_ppg * g for g in games]
    ax2.plot(games, league_pace, color='#f0e68c', linestyle='--', linewidth=1,
             alpha=0.6, label=f'League avg pace ({league_ppg:.2f} pt/gm)')

    ax2.fill_between(games, cumulative, alpha=0.3, color=color)
    ax2.plot(games, cumulative, color=color, linewidth=1.5, label='Cumulative pts')

    ax2.set_xlim(1, max(82, len(seq)))
    ax2.set_xlabel('Game Number', fontsize=10)
    ax2.set_ylabel('Cumulative Points', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=9, framealpha=0.3)

    plt.tight_layout()
    fname = f'nhl_deepdive_{abbrev}_{label.replace("-","")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f"✅ Saved: {fname}")


# ── Examples — change abbrev/season as desired ────────────────────────────────
deep_dive('WPG', SEASON_2425)   # 2024-25 Presidents Trophy winner
deep_dive('COL', SEASON_2526)   # 2025-26 Presidents Trophy winner


In [ ]:
for season_key, summary_df, label in [
    (SEASON_2425, summary_2425, '2425'),
    (SEASON_2526, summary_2526, '2526'),
]:
    export = summary_df.copy()
    export['sequence'] = export['abbrev'].map(
        lambda a: str(all_sequences[season_key].get(a, []))
    )
    fname = f'nhl_{label}_summary.csv'
    export.to_csv(fname, index=False)
    print(f"✅ Exported: {fname}")

print()
print("2024-25 Consistency Leaders:")
print(summary_2425[['abbrev','name','points','descents','z_score','consistency_rating','playoff','presidents']]
      .sort_values('consistency_rating', ascending=False).head(10).to_string(index=False))

print()
print("2025-26 Consistency Leaders (in-progress):")
print(summary_2526[['abbrev','name','points','descents','z_score','consistency_rating','playoff','presidents']]
      .sort_values('consistency_rating', ascending=False).head(10).to_string(index=False))
